# Forecasting Revenue and Quantifying Program Impact

This notebook builds a robust monthly sales forecast and quantifies the impact of a program using segment-level and partner-level time series. It uses Prophet with a regressor and includes fallback approaches for robustness.

In [47]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
import logging

logging.basicConfig(level=logging.INFO)

In [ ]:
# Robust load_and_clean (v4): flexible date parsing and duplicate-collapse

def load_and_clean(partner_types_path, sales_path):
    # Read partner types and normalize column names
    pt = pd.read_csv(partner_types_path, dtype=str)
    pt.columns = [c.strip() for c in pt.columns]

    # Auto-detect partner, type, and deal date columns in partner_types
    partner_candidates = [c for c in pt.columns if 'partner' in c.lower() or 'matching' in c.lower() or 'company' in c.lower()]
    type_candidates = [c for c in pt.columns if 'type' in c.lower() or 'segment' in c.lower()]
    deal_candidates = [c for c in pt.columns if 'deal' in c.lower() or 'kick' in c.lower() or 'date' in c.lower()]

    col_map = {}
    if partner_candidates:
        col_map[partner_candidates[0]] = 'Partner'
    if type_candidates:
        col_map[type_candidates[0]] = 'Type'
    if deal_candidates:
        col_map[deal_candidates[0]] = 'dealCreateDate'

    pt = pt.rename(columns=col_map)

    # Ensure minimal columns exist
    if 'Partner' not in pt.columns:
        raise KeyError(f"No partner column found in partner_types.csv. Columns: {pt.columns.tolist()}")
    if 'Type' not in pt.columns:
        pt['Type'] = 'Others'

    # Read sales and normalize columns
    sales = pd.read_csv(sales_path, dtype=str)
    sales.columns = [c.strip() for c in sales.columns]

    partner_cols = [c for c in sales.columns if 'partner' in c.lower() or 'partner_id' in c.lower() or 'matching' in c.lower()]
    date_cols = [c for c in sales.columns if 'date' in c.lower() or 'invoice' in c.lower()]
    sales_cols = [c for c in sales.columns if 'sales' in c.lower() or 'amount' in c.lower() or 'revenue' in c.lower()]

    if not partner_cols or not date_cols or not sales_cols:
        raise KeyError(f"Could not auto-detect required sales columns. sales.csv columns: {sales.columns.tolist()}")

    # Rename primary detected columns
    sales = sales.rename(columns={partner_cols[0]: 'Partner', date_cols[0]: 'Invoice Date', sales_cols[0]: 'Sales'})

    # Helper to collapse duplicate-named columns by taking first non-null value left-to-right
    def collapse_duplicates(df, colname):
        # find all column indices with this name
        idxs = [i for i, c in enumerate(df.columns) if c == colname]
        if len(idxs) > 1:
            # Temporarily give duplicate columns unique names based on position
            cols = list(df.columns)
            temp_names = []
            for j, i in enumerate(idxs):
                temp = f"{colname}__dup{j}"
                cols[i] = temp
                temp_names.append(temp)
            df.columns = cols
            logging.info(f"Found duplicate columns for '{colname}': {len(idxs)} occurrences. Coalescing.")
            # Coalesce left-to-right
            df[colname] = df[temp_names].bfill(axis=1).iloc[:, 0]
            # Drop temporary columns
            df = df.drop(columns=temp_names)
        return df

    sales = collapse_duplicates(sales, 'Invoice Date')
    sales = collapse_duplicates(sales, 'Sales')
    sales = collapse_duplicates(sales, 'Partner')

    # Flexible date parser for a Series
    def parse_dates_flexible(series, name='date'):
        s = series.astype(str).str.strip()
        parsed = pd.to_datetime(s, format='%d/%m/%Y', errors='coerce', dayfirst=True)
        recovered = parsed.notna().sum()
        logging.info(f"{name}: recovered {recovered} rows with format %d/%m/%Y")
        # try ISO-like
        mask = parsed.isna()
        if mask.any():
            parsed_iso = pd.to_datetime(s[mask], errors='coerce')
            parsed.loc[mask] = parsed_iso
            recovered2 = parsed.notna().sum() - recovered
            if recovered2:
                logging.info(f"{name}: recovered {recovered2} additional rows with ISO/automatic parsing")
        # try Excel serial numbers for remaining
        mask = parsed.isna()
        if mask.any():
            cand = s[mask].str.replace('\\D', '', regex=True)
            # candidate numeric strings
            is_num = cand.str.len().between(1,7) & cand.str.match('^\\d+$')
            if is_num.any():
                try:
                    nums = cand[is_num].astype(float)
                    excel_dates = pd.to_datetime(nums, unit='d', origin='1899-12-30', errors='coerce')
                    parsed.loc[mask][is_num] = excel_dates.values
                    logging.info(f"{name}: parsed {is_num.sum()} Excel-serial dates")
                except Exception:
                    pass
        # final fallback using dateutil
        mask = parsed.isna()
        if mask.any():
            parsed.loc[mask] = pd.to_datetime(s[mask], errors='coerce', dayfirst=True)
            logging.info(f"{name}: final fallback parsed {parsed.notna().sum() - recovered} rows total")
        return parsed


In [49]:
# Function to build monthly sales series for each segment and total
def build_monthly_sales(sales, pt):
    merged = sales.merge(pt[['Partner', 'Type']], on='Partner', how='left')
    merged['Type'] = merged['Type'].fillna('Others')
    merged['MonthStart'] = merged['Invoice Date'].dt.to_period('M').dt.to_timestamp()
    segments = ['Target', 'Control', 'Others']
    monthly = {}
    for seg in segments:
        seg_sales = merged[merged['Type'] == seg]
        m = seg_sales.groupby('MonthStart')['Sales'].sum()
        monthly[seg] = m
    total = merged.groupby('MonthStart')['Sales'].sum()
    monthly['Total'] = total
    all_months = pd.date_range(merged['MonthStart'].min(), merged['MonthStart'].max(), freq='MS')
    for k in monthly:
        monthly[k] = monthly[k].reindex(all_months, fill_value=0)
    return monthly, all_months

In [50]:
# Function to build program regressor for Target segment
def build_program_regressor(months, pt):
    target_kickoffs = pt[pt['Type'] == 'Target']['dealCreateDate'].dropna()
    if target_kickoffs.empty:
        raise ValueError('No valid Target kickoff dates found.')
    kickoff = target_kickoffs.min()
    program_on = (months >= kickoff).astype(int)
    return program_on, kickoff

In [51]:
# Function to fit Prophet with regressor and forecast two scenarios
def fit_prophet_with_regressor(target_series, months, program_on):
    df = pd.DataFrame({'ds': months, 'y': target_series.values, 'program_on': program_on})
    model = Prophet(yearly_seasonality=True)
    model.add_regressor('program_on')
    model.fit(df)
    last_month = months.max()
    future_months = pd.date_range(last_month + pd.offsets.MonthBegin(), periods=12, freq='MS')
    future_df = pd.DataFrame({'ds': future_months})
    future_df['program_on'] = 0
    forecast_cf = model.predict(future_df)
    future_df['program_on'] = 1
    forecast_prog = model.predict(future_df)
    return forecast_cf, forecast_prog, future_months, model

In [52]:
# Function to apply fixed uplift rate (DID) as fallback
def forecast_two_scenarios(counterfactual_forecast, did_rate=0.057411022):
    program_forecast = counterfactual_forecast * (1 + did_rate)
    return program_forecast

In [ ]:
# Main workflow: load data, build series, fit models, plot and output results
partner_types_path = 'partner_types.csv'
sales_path = 'sales.csv'

pt, sales, dropped = load_and_clean(partner_types_path, sales_path)
monthly, all_months = build_monthly_sales(sales, pt)
program_on, kickoff = build_program_regressor(all_months, pt)

post_kickoff_months = (all_months >= kickoff).sum()
if post_kickoff_months < 6:
    logging.warning('Post-kickoff history is too short (<6 months) to estimate regressor effect reliably.')

try:
    forecast_cf, forecast_prog, future_months, model = fit_prophet_with_regressor(monthly['Target'], all_months, program_on)
    total_cf = forecast_cf['yhat'].sum()
    total_prog = forecast_prog['yhat'].sum()
    incremental = total_prog - total_cf
    print(f"2026 Target Revenue (Counterfactual): {total_cf:.2f}")
    print(f"2026 Target Revenue (Program): {total_prog:.2f}")
    print(f"Incremental Program Impact: {incremental:.2f}")
except Exception as e:
    logging.warning(f"Prophet regressor estimation failed: {e}. Using fixed uplift rate.")
    df = pd.DataFrame({'ds': all_months, 'y': monthly['Target'].values})
    model_base = Prophet(yearly_seasonality=True)
    model_base.fit(df)
    future_months = pd.date_range(all_months.max() + pd.offsets.MonthBegin(), periods=12, freq='MS')
    future_df = pd.DataFrame({'ds': future_months})
    forecast_base = model_base.predict(future_df)
    total_cf = forecast_base['yhat'].sum()
    total_prog = forecast_two_scenarios(forecast_base['yhat']).sum()
    incremental = total_prog - total_cf
    print(f"2026 Target Revenue (Counterfactual): {total_cf:.2f}")
    print(f"2026 Target Revenue (Program, DID uplift): {total_prog:.2f}")
    print(f"Incremental Program Impact (DID): {incremental:.2f}")

plt.figure(figsize=(12,6))
plt.plot(future_months, forecast_cf['yhat'] if 'forecast_cf' in locals() else forecast_base['yhat'], label='Counterfactual')
plt.plot(future_months, forecast_prog['yhat'] if 'forecast_prog' in locals() else forecast_two_scenarios(forecast_base['yhat']), label='Program Scenario')
plt.title('Target Segment: 2026 Forecasts')
plt.xlabel('Month')
plt.ylabel('Sales ($)')
plt.legend()
plt.show()

for seg in ['Control', 'Others']:
    df_seg = pd.DataFrame({'ds': all_months, 'y': monthly[seg].values})
    model_seg = Prophet(yearly_seasonality=True)
    model_seg.fit(df_seg)
    future_df = pd.DataFrame({'ds': future_months})
    forecast_seg = model_seg.predict(future_df)
    plt.plot(future_months, forecast_seg['yhat'], label=f'{seg} Forecast')
plt.title('Control & Others: 2026 Baseline Forecasts')
plt.xlabel('Month')
plt.ylabel('Sales ($)')
plt.legend()
plt.show()

INFO:root:Found duplicate columns for 'Invoice Date': 2 occurrences. Coalescing.
INFO:root:Invoice Date: recovered 0 rows with format %d/%m/%Y
C:\Users\minh.nguyen\AppData\Local\Temp\ipykernel_36240\3126050393.py:76: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed_iso = pd.to_datetime(s[mask], errors='coerce')
INFO:root:Invoice Date: recovered 994 additional rows with ISO/automatic parsing
C:\Users\minh.nguyen\AppData\Local\Temp\ipykernel_36240\3126050393.py:98: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed.loc[mask] = pd.to_datetime(s[mask], errors='coerce', dayfirst=True)
INFO:root:Invoice Date: final fallback parsed 1654 rows total
INFO:root:dealCreateDate: recovered 115 rows with format %d/%m/%Y

KeyError: 'Type'

In [ ]:
print("Rows after cleaning:", len(sales))
print(sales.dtypes)
print("Date range:", sales['Invoice Date'].min(), sales['Invoice Date'].max())
print("Sample rows:\n", sales.head(5).to_string(index=False))

print("\nDropped rows count:", len(dropped))
if len(dropped):
    print("Sample dropped rows (raw strings):")
    show_cols = [c for c in ['Invoice Date', 'Sales', 'Partner'] if c in dropped.columns]
    print(dropped[show_cols].head(20).to_string(index=False))
    if 'Invoice Date' in dropped.columns:
        print('\nTop raw Invoice Date values in dropped rows:')
        print(dropped['Invoice Date'].astype(str).value_counts().head(50).to_string())